Drive Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

os.listdir('/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty')

Path distinction

In [ ]:
data_set = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty"
image_og = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/images"
masks_set = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/masks"
image_thumbs = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/image_thumbs"

files = os.listdir(masks_set)

print("Number of files:", len(files))



print("Dataset :",data_set)
print("Images  :", image_og)
print("Masks   :", masks_set)
print("Thumbs  :", image_thumbs)

In [ ]:
prop_s= os.path.join(masks_set, "propeller")
paint_s = os.path.join(masks_set, "paint_peel")
segmentation_s = os.path.join(masks_set, "segmentation")
defect_s = os.path.join(masks_set, "defect")
saliency_s= os.path.join(masks_set, "saliency")
seachest_s = os.path.join(masks_set, "sea_chest_grating")
overboard_s = os.path.join(masks_set, "over_board_valves")
corrosion_s = os.path.join(masks_set, "corrosion")
marinegrowth_s = os.path.join(masks_set, "marine_growth")
shiphull_s = os.path.join(masks_set, "ship_hull")
bilgekeel_s= os.path.join(masks_set, "bilge_keel")
anode_s = os.path.join(masks_set, "anode")

In [ ]:
paths = {
    "propeller": prop_s,
    "paint_peel": paint_s,
    "segmentation": segmentation_s,
    "defect": defect_s,
    "saliency": saliency_s,
    "sea_chest_grating": seachest_s,
    "over_board_valves": overboard_s,
    "corrosion": corrosion_s,
    "marine_growth": marinegrowth_s,
    "ship_hull": shiphull_s,
    "bilge_keel": bilgekeel_s,
    "anode": anode_s
}

for name, path in paths.items():
    print(f"{name:20} -> {os.path.exists(path)}")

In [ ]:
sample_id = "image_0016"

print("Image:", os.path.exists(os.path.join(image_og, sample_id + ".jpg")))

print("Ship hull:", os.path.exists(os.path.join(shiphull_s, sample_id + ".bmp")))
print("Marine growth:", os.path.exists(os.path.join(marinegrowth_s, sample_id + ".bmp")))
print("Corrosion:", os.path.exists(os.path.join(corrosion_s, sample_id + ".bmp")))
print("Segmentation:", os.path.exists(os.path.join(segmentation_s, sample_id + ".png")))

In [ ]:
import os
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

Color distinction

In [ ]:
CLASS_COLORS = [
    (0, 0, 0),        # 0 - Void
    (0, 0, 255),      # 1 - Ship hull
    (0, 128, 0),      # 2 - Marine growth
    (0, 255, 255),    # 3 - Anode
    (64, 224, 208),   # 4 - Overboard valve
    (128, 0, 128),    # 5 - Propeller
    (255, 0, 0),      # 6 - Paint peel
    (255, 165, 0),    # 7 - Bilge keel
    (255, 192, 203),  # 8 - Defect
    (255, 255, 0),    # 9 - Corrosion
    (255, 255, 255),  # 10 - Sea chest grating
]

Class convention

In [ ]:
import os
import shutil
import random
import pandas as pd
from PIL import Image
from tqdm import tqdm

# =========================
# PATHS
# =========================

data_set = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty"
image_og = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/images"
segmentation_s = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/masks/segmentation"
image_thumbs = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/image_thumbs"


split_csv = os.path.join(data_set, "train_test_split.csv")
Output_s = os.path.join(data_set, "preprocessed_512")

TARGET_SIZE = (512, 512)

# Reproducibility
random.seed(42)


# =========================
# LIACi CLASS COLORS
# =========================

CLASS_COLORS = {
    (0, 0, 0): 0,          # Void
    (0, 0, 255): 1,        # Ship hull
    (0, 128, 0): 2,        # Marine growth
    (0, 255, 255): 3,      # Anode
    (64, 224, 208): 4,     # Overboard valve
    (128, 0, 128): 5,      # Propeller
    (255, 0, 0): 6,        # Paint peel
    (255, 165, 0): 7,      # Bilge keel
    (255, 192, 203): 8,    # Defect
    (255, 255, 0): 9,      # Corrosion
    (255, 255, 255): 10    # Sea chest grating
}

NUM_CLASSES = 11

Read CSV

In [ ]:

df = pd.read_csv(split_csv)

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print("Official train:", len(train_df))
print("Official test :", len(test_df))
print("Not used      :", len(df[df["split"] == "not_used"]))

Split CSV

In [ ]:
train_images = train_df["file_name"].tolist()

random.shuffle(train_images)

val_size = 137

val_images = train_images[:val_size]
train_images = train_images[val_size:]

test_images = test_df["file_name"].tolist()

print("Final train:", len(train_images))
print("Validation :", len(val_images))
print("Test       :", len(test_images))

In [ ]:
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(Output_s, "images", split), exist_ok=True)
    os.makedirs(os.path.join(Output_s, "masks", split), exist_ok=True)

print("Output directory:", Output_s)

In [ ]:
import numpy as np

def convert_mask(mask_path):
    mask = Image.open(mask_path).convert("RGB")

    # Resize mask using NEAREST
    # Never use bilinear/bicubic for segmentation labels
    mask = mask.resize(TARGET_SIZE, Image.Resampling.NEAREST)

    mask = np.array(mask)

    label_mask = np.zeros(
        (mask.shape[0], mask.shape[1]),
        dtype=np.uint8
    )

    for color, class_id in CLASS_COLORS.items():
        matches = np.all(mask == color, axis=-1)
        label_mask[matches] = class_id

    return Image.fromarray(label_mask)

OUTPUT_DIR= "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/preprocessed_512"

In [ ]:
def preprocess_split(image_list, split_name):

    image_output = os.path.join(
        OUTPUT_DIR, "images", split_name
    )

    mask_output = os.path.join(
        OUTPUT_DIR, "masks", split_name
    )

    for filename in tqdm(
        image_list,
        desc=f"Processing {split_name}"
    ):

        # Remove extension safely
        base_name = os.path.splitext(filename)[0]

        image_path = os.path.join(
            image_og,
            filename
        )

        mask_path = os.path.join(
            segmentation_s,
            base_name + ".png"
        )

        # Check files
        if not os.path.exists(image_path):
            print("Missing image:", image_path)
            continue

        if not os.path.exists(mask_path):
            print("Missing mask:", mask_path)
            continue

        # -------------------------
        # IMAGE
        # -------------------------

        image = Image.open(image_path).convert("RGB")

        image = image.resize(
            TARGET_SIZE,
            Image.Resampling.BILINEAR
        )

        image.save(
            os.path.join(
                image_output,
                base_name + ".jpg"
            ),
            quality=95
        )

        # -------------------------
        # MASK
        # -------------------------

        mask = convert_mask(mask_path)

        mask.save(
            os.path.join(
                mask_output,
                base_name + ".png"
            )
        )

In [ ]:

preprocess_split(
    train_images,
    "train"
)

preprocess_split(
    val_images,
    "val"
)

preprocess_split(
    test_images,
    "test"
)

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image


class LIACiDataset(Dataset):

    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image_name = self.images[index]
        base_name = os.path.splitext(image_name)[0]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        mask_path = os.path.join(
            self.mask_dir,
            base_name + ".png"
        )

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Load mask
        mask = Image.open(mask_path)

        # Convert to NumPy
        image = np.array(image)
        mask = np.array(mask)

        # Image: HWC → CHW and normalize
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        # Mask: keep as class IDs
        mask = torch.from_numpy(mask).long()

        return image, mask

In [ ]:
import os
from torch.utils.data import DataLoader

DATA_ROOT = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_dataset_pretty/preprocessed_512"

TRAIN_IMAGES = os.path.join(DATA_ROOT, "images", "train")
TRAIN_MASKS  = os.path.join(DATA_ROOT, "masks", "train")

VAL_IMAGES = os.path.join(DATA_ROOT, "images", "val")
VAL_MASKS  = os.path.join(DATA_ROOT, "masks", "val")

TEST_IMAGES = os.path.join(DATA_ROOT, "images", "test")
TEST_MASKS  = os.path.join(DATA_ROOT, "masks", "test")


train_dataset = LIACiDataset(
    TRAIN_IMAGES,
    TRAIN_MASKS
)

val_dataset = LIACiDataset(
    VAL_IMAGES,
    VAL_MASKS
)

test_dataset = LIACiDataset(
    TEST_IMAGES,
    TEST_MASKS
)


# T4 cannot comfortably handle batch size 8
BATCH_SIZE = 4


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

print("\\nBatch size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
train_dataset = LIACiDataset(
    os.path.join(OUTPUT_DIR, "images", "train"),
    os.path.join(OUTPUT_DIR, "masks", "train")
)

print("Dataset size:", len(train_dataset))

In [ ]:
image, mask = train_dataset[0]

print("Image shape:", image.shape)
print("Image dtype:", image.dtype)

print("Mask shape:", mask.shape)
print("Mask dtype:", mask.dtype)

print("Mask classes:", torch.unique(mask))

In [ ]:
!pip install -q timm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

In [ ]:
class ASPP(nn.Module):

    def __init__(self, in_channels, out_channels=256):
        super().__init__()

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        self.branch2 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=12,
                dilation=12,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        self.branch3 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=24,
                dilation=24,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        self.branch4 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=36,
                dilation=36,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        self.project = nn.Sequential(
            nn.Conv2d(
                out_channels * 4,
                out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):

        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x4 = self.branch4(x)

        x = torch.cat([x1, x2, x3, x4], dim=1)

        return self.project(x)

In [ ]:
class DeepLabV3PlusDecoder(nn.Module):

    def __init__(
        self,
        low_level_channels=128,
        high_level_channels=2048,
        num_classes=11
    ):
        super().__init__()

        # Reduce low-level feature channels
        self.low_level = nn.Sequential(
            nn.Conv2d(
                low_level_channels,
                48,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True)
        )

        # ASPP
        self.aspp = ASPP(
            high_level_channels,
            256
        )

        # Decoder
        self.decoder = nn.Sequential(

            nn.Conv2d(
                256 + 48,
                256,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.classifier = nn.Conv2d(
            256,
            num_classes,
            kernel_size=1
        )

    def forward(self, low_level_feature, high_level_feature):

        # ASPP
        high_level_feature = self.aspp(high_level_feature)

        # Reduce low-level channels
        low_level_feature = self.low_level(
            low_level_feature
        )

        # Match spatial resolution
        high_level_feature = F.interpolate(
            high_level_feature,
            size=low_level_feature.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        # Combine
        x = torch.cat(
            [high_level_feature, low_level_feature],
            dim=1
        )

        x = self.decoder(x)

        x = self.classifier(x)

        return x

In [ ]:
class DeepLabV3PlusXception(nn.Module):

    def __init__(
        self,
        num_classes=11,
        pretrained=True
    ):
        super().__init__()

        self.backbone = timm.create_model(
            "xception41",
            pretrained=pretrained,
            features_only=True,
            output_stride=8
        )

        channels = self.backbone.feature_info.channels()

        print("Xception feature channels:", channels)

        self.decoder = DeepLabV3PlusDecoder(
            low_level_channels=channels[-4],
            high_level_channels=channels[-1],
            num_classes=num_classes
        )

    def forward(self, x):

        input_size = x.shape[2:]

        features = self.backbone(x)

        low_level_feature = features[-4]
        high_level_feature = features[-1]

        x = self.decoder(
            low_level_feature,
            high_level_feature
        )

        # Restore to original image size
        x = F.interpolate(
            x,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        return x

In [ ]:
NUM_CLASSES = 11

model = DeepLabV3PlusXception(
    num_classes=NUM_CLASSES,
    pretrained=True
)

In [ ]:

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)
print("Fresh DeepLabV3+ Xception model)")

In [ ]:

images, masks = next(iter(train_loader))

print("Input :", images.shape)
print("Mask  :", masks.shape)

Metrics

In [ ]:
import numpy as np
import torch


CLASS_NAMES = [
    "Void",
    "Ship Hull",
    "Marine Growth",
    "Anode",
    "Overboard Valve",
    "Propeller",
    "Paint Peel",
    "Bilge Keel",
    "Defect",
    "Corrosion",
    "Sea Chest Grating"
]


def calculate_metrics(predictions, targets, num_classes=11):

    predictions = predictions.argmax(dim=1)

    total_correct = 0
    total_pixels = 0

    class_iou = []
    class_precision = []
    class_recall = []
    class_f1 = []

    for class_id in range(num_classes):

        pred_class = predictions == class_id
        target_class = targets == class_id

        TP = (pred_class & target_class).sum().item()
        FP = (pred_class & ~target_class).sum().item()
        FN = (~pred_class & target_class).sum().item()

        union = TP + FP + FN

        # IoU
        if union > 0:
            iou = TP / union
        else:
            iou = float("nan")

        # Precision
        if TP + FP > 0:
            precision = TP / (TP + FP)
        else:
            precision = float("nan")

        # Recall
        if TP + FN > 0:
            recall = TP / (TP + FN)
        else:
            recall = float("nan")

        # F1
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = float("nan")

        class_iou.append(iou)
        class_precision.append(precision)
        class_recall.append(recall)
        class_f1.append(f1)

    # Pixel accuracy
    total_correct = (predictions == targets).sum().item()
    total_pixels = targets.numel()

    accuracy = total_correct / total_pixels

    # Mean metrics
    miou = np.nanmean(class_iou)
    mean_precision = np.nanmean(class_precision)
    mean_recall = np.nanmean(class_recall)
    mean_f1 = np.nanmean(class_f1)

    # Corrosion = class 9
    corrosion_iou = class_iou[9]
    corrosion_precision = class_precision[9]
    corrosion_recall = class_recall[9]
    corrosion_f1 = class_f1[9]

    return {
        "accuracy": accuracy,
        "precision": mean_precision,
        "recall": mean_recall,
        "f1": mean_f1,
        "miou": miou,
        "class_iou": class_iou,
        "class_precision": class_precision,
        "class_recall": class_recall,
        "class_f1": class_f1,
        "corrosion_iou": corrosion_iou,
        "corrosion_precision": corrosion_precision,
        "corrosion_recall": corrosion_recall,
        "corrosion_f1": corrosion_f1
    }

Checkpoint Directory

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Checkpoint directory:", CHECKPOINT_DIR)

In [ ]:
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Directory exists:", os.path.exists(CHECKPOINT_DIR))

1 epoch Trainer Function

In [ ]:
from tqdm import tqdm

def train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device
):
    model.train()

    running_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc="Training",
        leave=False
    )

    for images, masks in progress_bar:

        images = images.to(device)
        masks = masks.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, masks)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Track loss
        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_loss = running_loss / len(train_loader)

    return average_loss

Validation Function

In [ ]:
def validate_one_epoch(
    model,
    val_loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        progress_bar = tqdm(
            val_loader,
            desc="Validation",
            leave=False
        )

        for images, masks in progress_bar:

            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            outputs = model(images)

            # Validation loss
            loss = criterion(outputs, masks)
            running_loss += loss.item()

            # Store predictions and masks
            all_predictions.append(
                outputs.cpu()
            )

            all_targets.append(
                masks.cpu()
            )

    average_loss = running_loss / len(val_loader)

    # Combine all validation batches
    all_predictions = torch.cat(
        all_predictions,
        dim=0
    )

    all_targets = torch.cat(
        all_targets,
        dim=0
    )

    # Calculate metrics
    metrics = calculate_metrics(
        all_predictions,
        all_targets,
        num_classes=11
    )

    metrics["loss"] = average_loss

    return metrics

CheckPoint-2

In [ ]:
import os

# ==========================================
# RUN 2 CHECKPOINT DIRECTORY
# ==========================================

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "LIACi_checkpoints_run2"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "latest.pth"
)

print("Checkpoint directory:")
print(CHECKPOINT_DIR)

print(
    "\\nExisting latest checkpoint:",
    os.path.exists(LATEST_CHECKPOINT)
)

In [ ]:
import torch

# ==========================================
# LOSS FUNCTION
# ==========================================

criterion = torch.nn.CrossEntropyLoss()


# ==========================================
# OPTIMIZER
# ==========================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)


# ==========================================
# LEARNING RATE SCHEDULER
# ==========================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)


print("Loss      :", type(criterion).__name__)
print("Mode      :", scheduler.mode)
print("Optimizer :", type(optimizer).__name__)
print("LR        :", optimizer.param_groups[0]["lr"])
print("Scheduler :", type(scheduler).__name__)

In [ ]:
LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "latest.pth"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pth"
)

print("Latest:", LATEST_CHECKPOINT)
print("Best  :", BEST_CHECKPOINT)

save checkpoint

In [ ]:
def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    best_miou,
    filename
):
    checkpoint = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_miou": float(best_miou)
    }

    path = os.path.join(
        CHECKPOINT_DIR,
        filename
    )

    torch.save(
        checkpoint,
        path
    )

    print(f"Checkpoint saved: {path}")

CheckPoint Loader

In [ ]:
def load_checkpoint(
    model,
    optimizer,
    scheduler,
    checkpoint_path
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    # Restore model weights
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    # Restore optimizer
    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # Restore learning-rate scheduler
    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    # Continue AFTER the last completed epoch
    start_epoch = int(checkpoint["epoch"]) + 1

    # Restore the REAL best mIoU achieved so far
    best_miou = float(checkpoint["best_miou"])

    print(f"Checkpoint loaded: {checkpoint_path}")
    print(f"Last completed epoch: {checkpoint['epoch'] + 1}")
    print(f"Next epoch: {start_epoch + 1}")
    print(f"Best mIoU so far: {best_miou:.4f}")

    return start_epoch, best_miou

In [ ]:
# ==========================================
# RUN 2 TRAINING STATE
# ==========================================

EPOCHS = 50

start_epoch = 0
best_miou = -1.0


# ==========================================
# RESUME ONLY FROM RUN 2
# ==========================================

if os.path.exists(LATEST_CHECKPOINT):

    print("Run 2 checkpoint found.")

    start_epoch, best_miou = load_checkpoint(
        model,
        optimizer,
        scheduler,
        LATEST_CHECKPOINT
    )

else:

    print("No Run 2 checkpoint found.")
    print("Starting fresh from epoch 1.")


print("\\nTraining configuration")
print("Total epochs :", EPOCHS)
print("Start epoch  :", start_epoch + 1)
print("Best mIoU    :", best_miou)

In [ ]:
print("=" * 60)
print("RUN 2 — FINAL PRE-FLIGHT CHECK")
print("=" * 60)

print("Device              :", device)
print("GPU                 :", torch.cuda.get_device_name(0))

print("\\nDataset")
print("Train samples       :", len(train_dataset))
print("Validation samples  :", len(val_dataset))
print("Test samples        :", len(test_dataset))
print("Batch size          :", train_loader.batch_size)

print("\\nModel")
print("Number of classes   :", NUM_CLASSES)

print("\\nTraining")
print("Epochs              :", EPOCHS)
print("Start epoch         :", start_epoch + 1)
print("Best mIoU           :", best_miou)
print("Loss                :", criterion)
print("Optimizer           :", type(optimizer).__name__)
print("Learning rate       :", optimizer.param_groups[0]["lr"])

print("\\nScheduler")
print("Scheduler           :", type(scheduler).__name__)
print("Mode                :", scheduler.mode)
print("Factor              :", scheduler.factor)
print("Patience            :", scheduler.patience)

print("\\nCheckpoint")
print("Directory           :", CHECKPOINT_DIR)
print("Latest path         :", LATEST_CHECKPOINT)
print("Latest exists       :", os.path.exists(LATEST_CHECKPOINT))

print("=" * 60)

50 Epoch's Trainer with checkpoints and validations

In [ ]:
import os
import math

EPOCHS = 50

# ============================================================
# RUN 2 CHECKPOINT PATH
# ============================================================

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "latest.pth"
)


# ============================================================
# RESUME SETTINGS
# ============================================================

start_epoch = 0
best_miou = -1.0


# Resume ONLY if Run 2 already has latest.pth
if os.path.exists(LATEST_CHECKPOINT):

    print("Run 2 checkpoint found.")

    start_epoch, best_miou = load_checkpoint(
        model,
        optimizer,
        scheduler,
        LATEST_CHECKPOINT
    )

else:

    print("No Run 2 checkpoint found.")
    print("Starting fresh training from epoch 1.")


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(start_epoch, EPOCHS):

    print("\n" + "=" * 55)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 55)


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )


    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------

    val_metrics = validate_one_epoch(
        model,
        val_loader,
        criterion,
        device
    )


    # --------------------------------------------------------
    # UPDATE LEARNING RATE
    # --------------------------------------------------------

    scheduler.step(
        val_metrics["miou"]
    )

    current_lr = optimizer.param_groups[0]["lr"]


    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_metrics['loss']:.4f}")

    print(f"Accuracy   : {val_metrics['accuracy']:.4f}")
    print(f"Precision  : {val_metrics['precision']:.4f}")
    print(f"Recall     : {val_metrics['recall']:.4f}")
    print(f"F1 Score   : {val_metrics['f1']:.4f}")
    print(f"mIoU       : {val_metrics['miou']:.4f}")


    # --------------------------------------------------------
    # CORROSION METRICS
    # Corrosion is one of the 11 ship-hull inspection classes
    # --------------------------------------------------------

    print("\nCorrosion Metrics")

    corrosion_iou = val_metrics["corrosion_iou"]
    corrosion_precision = val_metrics["corrosion_precision"]
    corrosion_recall = val_metrics["corrosion_recall"]
    corrosion_f1 = val_metrics["corrosion_f1"]

    print(
        "IoU       :",
        "N/A" if math.isnan(corrosion_iou)
        else f"{corrosion_iou:.4f}"
    )

    print(
        "Precision :",
        "N/A" if math.isnan(corrosion_precision)
        else f"{corrosion_precision:.4f}"
    )

    print(
        "Recall    :",
        "N/A" if math.isnan(corrosion_recall)
        else f"{corrosion_recall:.4f}"
    )

    print(
        "F1 Score  :",
        "N/A" if math.isnan(corrosion_f1)
        else f"{corrosion_f1:.4f}"
    )

    print(f"\\nLearning Rate: {current_lr:.8f}")


    # ========================================================
    # SAFE CHECKPOINT MANAGEMENT
    # IMPORTANT: THIS MUST REMAIN INSIDE THE FOR LOOP
    # ========================================================

    current_miou = float(
        val_metrics["miou"]
    )

    # Compare BEFORE updating best_miou
    is_best = current_miou > best_miou


    # --------------------------------------------------------
    # Update best score
    # --------------------------------------------------------

    if is_best:
        best_miou = current_miou


    # --------------------------------------------------------
    # ALWAYS SAVE LATEST
    #
    # latest.pth = resume checkpoint
    # It is allowed to be overwritten every epoch.
    # --------------------------------------------------------

    save_checkpoint(
        epoch,
        model,
        optimizer,
        scheduler,
        best_miou,
        "latest.pth"
    )


    # --------------------------------------------------------
    # SAVE EVERY NEW BEST PERMANENTLY
    #
    # These files NEVER overwrite one another.
    # --------------------------------------------------------

    if is_best:

        best_filename = (
            f"best_epoch_{epoch + 1:02d}_"
            f"miou_{current_miou:.4f}.pth"
        )

        save_checkpoint(
            epoch,
            model,
            optimizer,
            scheduler,
            best_miou,
            best_filename
        )

        print("\\n*** NEW BEST MODEL ***")
        print(f"Epoch : {epoch + 1}")
        print(f"mIoU  : {current_miou:.4f}")
        print(f"Saved : {best_filename}")


# ============================================================
# TRAINING FINISHED
# ============================================================

print("\nTraining complete.")
print(f"Best validation mIoU: {best_miou:.4f}")

CheckPoint Evaluation

In [ ]:
import os
import torch

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "latest.pth"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pth"
)


def inspect_checkpoint(path):

    print("\\n" + "=" * 55)
    print("Checkpoint:", os.path.basename(path))
    print("=" * 55)

    if not os.path.exists(path):
        print("Checkpoint does not exist.")
        return

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Saved epoch index :", checkpoint.get("epoch"))
    print("Displayed epoch   :", checkpoint.get("epoch", -1) + 1)
    print("Stored best mIoU  :", checkpoint.get("best_miou"))

    print(
      "Contains model    :",
        "model_state_dict" in checkpoint
    )

    print(
          "Contains optimizer:",
        "optimizer_state_dict" in checkpoint
    )

    print(
        "Contains scheduler:",
        "scheduler_state_dict" in checkpoint
    )

    # Remove checkpoint from RAM
    del checkpoint


inspect_checkpoint(LATEST_CHECKPOINT)
inspect_checkpoint(BEST_CHECKPOINT)

Save Baseline model

In [ ]:
import os
import shutil

CHECKPOINT_DIR = "/content/drive/MyDrive/PEP_SEG_DATA/LIACi_checkpoints_run2"

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_epoch_38_miou_0.5508.pth"
)

FINAL_MODEL_DIR = "/content/drive/MyDrive/PEP_SEG_DATA/DeepLabV3Plus_Xception_Final"
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

FINAL_CHECKPOINT = os.path.join(
    FINAL_MODEL_DIR,
    "deeplabv3plus_xception_best.pth"
)

# Safety check
assert os.path.exists(BEST_CHECKPOINT), \
    f"Best checkpoint not found: {BEST_CHECKPOINT}"

# COPY — don't move/delete the original
shutil.copy2(BEST_CHECKPOINT, FINAL_CHECKPOINT)

print("Best checkpoint found :", BEST_CHECKPOINT)
print("Final model saved to  :", FINAL_CHECKPOINT)
print("File size (MB)         :", os.path.getsize(FINAL_CHECKPOINT) / (1024**2))

Testing Evalution

load trained model for testing

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FINAL_CHECKPOINT = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "DeepLabV3Plus_Xception_Final/"
    "deeplabv3plus_xception_best.pth"
)

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

model = model.to(device)
model.eval()

print("=" * 50)
print("BEST MODEL LOADED FOR TESTING")
print("=" * 50)
print("Epoch        :", checkpoint["epoch"] + 1)
print("Val mIoU     :", checkpoint["best_miou"])
print("Device       :", device)
print("Training mode:", model.training)

assert checkpoint["epoch"] + 1 == 38
assert model.training is False


print("Epoch 38 weights loaded")
print("Model is in evaluation mode")
print("Ready for TEST SET")


CP2 test set

In [ ]:
from tqdm import tqdm

NUM_CLASSES = 11

CLASS_NAMES = [
    "Void",
    "Ship Hull",
    "Marine Growth",
    "Anode",
    "Overboard Valve",
    "Propeller",
    "Paint Peel",
    "Bilge Keel",
    "Defect",
    "Corrosion",
    "Sea Chest Grating"
]

# ---------------------------------------------------------
# FINAL TEST EVALUATION
# ---------------------------------------------------------

model.eval()

confusion_matrix = np.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=np.int64
)

total_test_loss = 0.0

# Save predictions + ground truths for later visualization
all_predictions = []
all_targets = []

with torch.no_grad():

    progress_bar = tqdm(
        test_loader,
        desc="Final Testing"
    )

    for images, masks in progress_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        # Forward pass
        outputs = model(images)

        # Test loss
        loss = criterion(outputs, masks)
        total_test_loss += loss.item()

        # Convert logits -> class predictions
        predictions = torch.argmax(
            outputs,
            dim=1
        )

        # Move to CPU
        preds_np = predictions.cpu().numpy()
        masks_np = masks.cpu().numpy()

        # Save for later visualization
        all_predictions.append(preds_np)
        all_targets.append(masks_np)

        # -------------------------------------------------
        # CONFUSION MATRIX
        # rows    = Ground Truth
        # columns = Prediction
        # -------------------------------------------------

        valid = (
            (masks_np >= 0) &
            (masks_np < NUM_CLASSES)
        )

        indices = (
            NUM_CLASSES * masks_np[valid].astype(int)
            + preds_np[valid].astype(int)
        )

        confusion_matrix += np.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        ).reshape(NUM_CLASSES, NUM_CLASSES)


# ---------------------------------------------------------
# LOSS
# ---------------------------------------------------------

test_loss = total_test_loss / len(test_loader)


# ---------------------------------------------------------
# METRICS FROM CONFUSION MATRIX
# ---------------------------------------------------------

TP = np.diag(confusion_matrix)

FP = confusion_matrix.sum(axis=0) - TP

FN = confusion_matrix.sum(axis=1) - TP

total_pixels = confusion_matrix.sum()


# Pixel Accuracy
accuracy = TP.sum() / total_pixels


# Precision
precision = np.divide(
    TP,
    TP + FP,
    out=np.full(NUM_CLASSES, np.nan, dtype=float),
    where=(TP + FP) != 0
)


# Recall
recall = np.divide(
    TP,
    TP + FN,
    out=np.full(NUM_CLASSES, np.nan, dtype=float),
    where=(TP + FN) != 0
)


# IoU
iou = np.divide(
    TP,
    TP + FP + FN,
    out=np.full(NUM_CLASSES, np.nan, dtype=float),
    where=(TP + FP + FN) != 0
)


# F1 Score
f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.full(NUM_CLASSES, np.nan, dtype=float),
    where=(precision + recall) != 0
)


# Macro metrics
mean_precision = np.nanmean(precision)
mean_recall = np.nanmean(recall)
mean_f1 = np.nanmean(f1)
miou = np.nanmean(iou)


# ---------------------------------------------------------
# DISPLAY FINAL RESULTS
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST RESULTS — DeepLabV3+ Xception")
print("=" * 70)

print(f"Test Loss       : {test_loss:.4f}")
print(f"Pixel Accuracy  : {accuracy:.4f}")
print(f"Mean Precision  : {mean_precision:.4f}")
print(f"Mean Recall     : {mean_recall:.4f}")
print(f"Mean F1 Score   : {mean_f1:.4f}")
print(f"Mean IoU        : {miou:.4f}")

print("\\n" + "=" * 70)
print("PER-CLASS TEST RESULTS")
print("=" * 70)

print(
    f"{'Class':<22}"
    f"{'IoU':>10}"
    f"{'Precision':>12}"
    f"{'Recall':>10}"
    f"{'F1':>10}"
)

print("-" * 64)

for i, class_name in enumerate(CLASS_NAMES):

    print(
        f"{class_name:<22}"
        f"{iou[i]:>10.4f}"
        f"{precision[i]:>12.4f}"
        f"{recall[i]:>10.4f}"
        f"{f1[i]:>10.4f}"
    )

print("\n" + "=" * 70)
print("CORROSION TEST PERFORMANCE")
print("=" * 70)

corrosion_id = 9

print(f"IoU       : {iou[corrosion_id]:.4f}")
print(f"Precision : {precision[corrosion_id]:.4f}")
print(f"Recall    : {recall[corrosion_id]:.4f}")
print(f"F1 Score  : {f1[corrosion_id]:.4f}")

print("\nConfusion Matrix Shape :", confusion_matrix.shape)
print("Total Evaluated Pixels :", total_pixels)

print("\n" + "=" * 70)
print("✓ FINAL TESTING COMPLETE")
print("=" * 70)

CP2 Confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

CLASS_NAMES = [
    "Void",
    "Ship Hull",
    "Marine Growth",
    "Anode",
    "Overboard Valve",
    "Propeller",
    "Paint Peel",
    "Bilge Keel",
    "Defect",
    "Corrosion",
    "Sea Chest Grating"
]

FINAL_MODEL_DIR = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "DeepLabV3Plus_Xception_Final"
)

# =========================================================
# NORMALIZE CONFUSION MATRIX BY GROUND-TRUTH CLASS
# =========================================================

row_sums = confusion_matrix.sum(axis=1, keepdims=True)

cm_normalized = np.divide(
    confusion_matrix,
    row_sums,
    out=np.zeros_like(confusion_matrix, dtype=float),
    where=row_sums != 0
)

# Convert to percentage
cm_percentage = cm_normalized * 100


# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(13, 11))

im = ax.imshow(
    cm_percentage,
    interpolation="nearest",
    cmap="Blues",
    vmin=0,
    vmax=100
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Percentage (%)", fontsize=12)

ax.set(
    xticks=np.arange(len(CLASS_NAMES)),
    yticks=np.arange(len(CLASS_NAMES)),
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ylabel="Ground Truth",
    xlabel="Predicted Class",
    title="Normalized Confusion Matrix — DeepLabV3+ Xception"
)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor"
)

# =========================================================
# ADD VALUES TO CELLS
# =========================================================

threshold = 50

for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):

        value = cm_percentage[i, j]

        # Don't clutter matrix with tiny values
        if value >= 1.0:
            ax.text(
                j,
                i,
                f"{value:.1f}",
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
                fontsize=8
            )

plt.tight_layout()


# =========================================================
# SAVE
# =========================================================

save_path = os.path.join(
    FINAL_MODEL_DIR,
    "confusion_matrix_normalized.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(save_path)

CSV file to store values

In [ ]:
import pandas as pd
import os

FINAL_MODEL_DIR = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "DeepLabV3Plus_Xception_Final"
)

results_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "IoU": iou,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

results_path = os.path.join(
    FINAL_MODEL_DIR,
    "test_per_class_metrics.csv"
)

results_df.to_csv(results_path, index=False)

print(results_df.round(4))

print("\nSaved:")
print(results_path)

Per-Class Graph

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

FINAL_MODEL_DIR = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "DeepLabV3Plus_Xception_Final"
)

x = np.arange(len(CLASS_NAMES))
width = 0.20

fig, ax = plt.subplots(figsize=(16, 8))

ax.bar(x - 1.5*width, iou,       width, label="IoU")
ax.bar(x - 0.5*width, precision, width, label="Precision")
ax.bar(x + 0.5*width, recall,    width, label="Recall")
ax.bar(x + 1.5*width, f1,        width, label="F1 Score")

ax.set_title(
    "Per-Class Performance — DeepLabV3+ Xception",
    fontsize=16
)

ax.set_ylabel("Score")
ax.set_xlabel("Class")
ax.set_ylim(0, 1)

ax.set_xticks(x)
ax.set_xticklabels(
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

ax.legend()

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()

save_path = os.path.join(
    FINAL_MODEL_DIR,
    "per_class_performance.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(save_path)

Visual Qualitative results

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import random

FINAL_MODEL_DIR = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "DeepLabV3Plus_Xception_Final"
)

# =========================================================
# CLASS COLORS
# =========================================================

CLASS_COLORS = np.array([
    [0,   0,   0],       # 0  Void
    [0,   0,   255],     # 1  Ship Hull
    [0,   128, 0],       # 2  Marine Growth
    [0,   255, 255],     # 3  Anode
    [64,  224, 208],     # 4  Overboard Valve
    [128, 0,   128],     # 5  Propeller
    [255, 0,   0],       # 6  Paint Peel
    [255, 165, 0],       # 7  Bilge Keel
    [255, 192, 203],     # 8  Defect
    [255, 255, 0],       # 9  Corrosion
    [255, 255, 255]      # 10 Sea Chest Grating
], dtype=np.uint8)


# =========================================================
# CONVERT CLASS MASK -> RGB IMAGE
# =========================================================

def mask_to_rgb(mask):
    return CLASS_COLORS[mask]


# =========================================================
# DENORMALIZE IMAGE
# =========================================================

def denormalize_image(image):

    image = image.cpu().numpy().transpose(1, 2, 0)

    # ImageNet normalization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = image * std + mean
    image = np.clip(image, 0, 1)

    return image


# =========================================================
# SELECT TEST IMAGES
# =========================================================

NUM_SAMPLES = 5

indices = np.random.choice(
    len(test_dataset),
    size=NUM_SAMPLES,
    replace=False
)

print("Random test indices:", indices)


# =========================================================
# VISUALIZATION
# =========================================================

model.eval()

fig, axes = plt.subplots(
    NUM_SAMPLES,
    3,
    figsize=(15, NUM_SAMPLES * 5)
)

with torch.no_grad():

    for row, idx in enumerate(indices):

        image, mask = test_dataset[idx]

        input_tensor = image.unsqueeze(0).to(device)

        output = model(input_tensor)

        prediction = torch.argmax(
            output,
            dim=1
        )[0].cpu().numpy()

        original = denormalize_image(image)

        gt_rgb = mask_to_rgb(
            mask.cpu().numpy()
        )

        pred_rgb = mask_to_rgb(
            prediction
        )

        # ORIGINAL
        axes[row, 0].imshow(original)
        axes[row, 0].set_title(
            f"Original — Test #{idx}"
        )

        # GROUND TRUTH
        axes[row, 1].imshow(gt_rgb)
        axes[row, 1].set_title(
            "Ground Truth"
        )

        # PREDICTION
        axes[row, 2].imshow(pred_rgb)
        axes[row, 2].set_title(
            "Prediction"
        )

        for col in range(3):
            axes[row, col].axis("off")


plt.suptitle(
    "Qualitative Segmentation Results — DeepLabV3+ Xception",
    fontsize=18
)

plt.tight_layout()

save_path = os.path.join(
    FINAL_MODEL_DIR,
    "qualitative_segmentation_results.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(save_path)

Correlation HeatMap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

FINAL_MODEL_DIR = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "DeepLabV3Plus_Xception_Final"
)

# ==========================================
# SELECT METRICS
# ==========================================

metric_data = results_df[
    ["IoU", "Precision", "Recall", "F1"]
]

# Pearson correlation
correlation_matrix = metric_data.corr(method="pearson")

print("Correlation Matrix:")
print(correlation_matrix.round(4))


# ==========================================
# PLOT HEATMAP
# ==========================================

fig, ax = plt.subplots(figsize=(8, 7))

im = ax.imshow(
    correlation_matrix.values,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

# Color bar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(
    "Pearson Correlation Coefficient",
    fontsize=11
)

metrics = correlation_matrix.columns

ax.set_xticks(np.arange(len(metrics)))
ax.set_yticks(np.arange(len(metrics)))

ax.set_xticklabels(metrics)
ax.set_yticklabels(metrics)

ax.set_title(
    "Correlation Between Segmentation Performance Metrics",
    fontsize=14,
    pad=15
)

# ==========================================
# DISPLAY CORRELATION VALUES
# ==========================================

for i in range(len(metrics)):
    for j in range(len(metrics)):

        value = correlation_matrix.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.3f}",
            ha="center",
            va="center",
            fontsize=11,
            color="white" if abs(value) > 0.6 else "black"
        )

plt.tight_layout()


# ==========================================
# SAVE
# ==========================================

save_path = os.path.join(
    FINAL_MODEL_DIR,
    "metric_correlation_heatmap.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(save_path)

In [ ]:
%cd /content

!git clone https://github.com/Akashgks/Underwater_Ship_Hull_Inspection_Semantic_Segmentation.git

%cd /content/Underwater_Ship_Hull_Inspection_Semantic_Segmentation

In [ ]:
import os

REPO_DIR = "/content/Underwater_Ship_Hull_Inspection_Semantic_Segmentation"

os.chdir(REPO_DIR)

# Create project folders
folders = [
    "notebooks",
    "src",
    "results",
    "docs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Create .gitignore
gitignore = """
# ==============================
# Model checkpoints
# ==============================
*.pth
*.pt
*.ckpt

# ==============================
# Dataset
# ==============================
data/
dataset/
LIACi_dataset_pretty/
preprocessed_512/

# ==============================
# Python
# ==============================
__pycache__/
*.pyc
*.pyo

# ==============================
# Jupyter / Colab
# ==============================
.ipynb_checkpoints/

# ==============================
# Temporary files
# ==============================
*.tmp
*.log

# ==============================
# OS files
# ==============================
.DS_Store
Thumbs.db
"""

with open(".gitignore", "w") as f:
    f.write(gitignore.strip() + "\\n")

print("Repository:", os.getcwd())
print("\\nFolders created:")
for folder in folders:
    print(" -", folder)

print("\\n✓ .gitignore created")

In [ ]:
!git status

In [ ]:
import os
import shutil

SOURCE_DIR = (
    "/content/drive/MyDrive/PEP_SEG_DATA/"
    "DeepLabV3Plus_Xception_Final"
)

REPO_DIR = (
    "/content/Underwater_Ship_Hull_Inspection_"
    "Semantic_Segmentation"
)

RESULTS_DIR = os.path.join(REPO_DIR, "results")

files_to_copy = [
    "confusion_matrix_normalized.png",
    "per_class_performance.png",
    "qualitative_segmentation_results.png",
    "metric_correlation_heatmap.png",
    "test_per_class_metrics.csv",
    "test_dice_scores.csv"
]

print("=" * 60)
print("COPYING PROJECT RESULTS")
print("=" * 60)

for filename in files_to_copy:

    source = os.path.join(SOURCE_DIR, filename)
    destination = os.path.join(RESULTS_DIR, filename)

    if os.path.exists(source):

        shutil.copy2(source, destination)

        print("✓", filename)

    else:

        print("SKIPPED — not created yet:", filename)

print("=" * 60)
print("✓ COPY COMPLETE")

In [ ]:
%cd /content/Underwater_Ship_Hull_Inspection_Semantic_Segmentation

!git status

In [ ]:
!find . -type f -size +50M -print

In [ ]:
!find . -maxdepth 2 -type f -print

In [ ]:
import os
import shutil

source = "/content/drive/MyDrive/Colab Notebooks/PEP_Segmentation_On_Ship_Hull(DeepLabV3+).ipynb"

destination = (
    "/content/Underwater_Ship_Hull_Inspection_Semantic_Segmentation/"
    "notebooks/DeepLabV3Plus_Xception_LIACi.ipynb"
)

assert os.path.isfile(source), "Source notebook not found!"

shutil.copy2(source, destination)

print("✓ Notebook copied successfully")
print("Size:", round(os.path.getsize(destination) / 1024 / 1024, 2), "MB")
print("Location:", destination)

In [ ]:
import json

with open(destination, encoding="utf-8") as f:
    notebook = json.load(f)

print("Total cells:", len(notebook["cells"]))
print("Code cells:", sum(c["cell_type"] == "code" for c in notebook["cells"]))
print("Notebook valid:", notebook.get("nbformat") == 4)

In [ ]:
import json
import os
import re

notebook_path = (
    "/content/Underwater_Ship_Hull_Inspection_Semantic_Segmentation/"
    "notebooks/DeepLabV3Plus_Xception_LIACi.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Total cells:", len(nb["cells"]))
print("Code cells:", sum(c["cell_type"] == "code" for c in nb["cells"]))

# Check source code for potentially sensitive information
patterns = [
    r"ghp_[A-Za-z0-9_]+",
    r"github_pat_[A-Za-z0-9_]+",
    r"AIza[A-Za-z0-9_-]+",
    r"sk-[A-Za-z0-9_-]{20,\}",
    r"(?i)(?:password|api_key|access_token|secret)\s*=\s*['\"][^'\"]+['\"]"
]

matches = []

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    for pattern in patterns:
        if re.search(pattern, source):
            matches.append(i + 1)

print("Potential sensitive cells:", sorted(set(matches)))

if matches:
    print("Review these cells before publishing. No changes made.")
else:
    # Clear stored outputs
    for cell in nb["cells"]:
        if cell["cell_type"] == "code":
            cell["outputs"] = []
            cell["execution_count"] = None

    with open(notebook_path, "w", encoding="utf-8") as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)

    print("✓ Notebook outputs cleared")
    print("✓ Original Drive notebook untouched")
    print("✓ Clean notebook size:",
          round(os.path.getsize(notebook_path) / 1024 / 1024, 2), "MB")

In [ ]:
from pathlib import Path

repo = Path(
    "/content/Underwater_Ship_Hull_Inspection_Semantic_Segmentation"
)

readme = """# Underwater Ship Hull Inspection — Semantic Segmentation

Multi-class semantic segmentation of underwater ship-hull inspection images using **DeepLabV3+ with an Xception backbone**, trained on the LIACi dataset.

## Project Overview

This project segments underwater inspection images into 11 semantic classes, covering ship structures, marine growth, corrosion, paint peeling, and other surface conditions.

## Architecture

- Model: DeepLabV3+
- Backbone: Xception
- Pretrained backbone: ImageNet
- Output stride: 8
- ASPP dilation rates: 12, 24, 36
- Input resolution: 512 × 512
- Output classes: 11

## Dataset

**LIACi — Large-scale Image Annotation for Underwater Inspection**

Dataset: https://liaci.sintef.cloud/

Classes:
1. Void
2. Ship Hull
3. Marine Growth
4. Anode
5. Overboard Valve
6. Propeller
7. Paint Peel
8. Bilge Keel
9. Defect
10. Corrosion
11. Sea Chest Grating

## Training Configuration

| Parameter | Value |
|---|---|
| Framework | PyTorch |
| Optimizer | AdamW |
| Initial learning rate | 0.0001 |
| Loss function | Cross-Entropy |
| Batch size | 4 |
| Epochs | 50 |
| Scheduler | ReduceLROnPlateau |
| Hardware | NVIDIA Tesla T4 |
| Best checkpoint | Epoch 38 |

## Final Test Results

Evaluated on 191 held-out test images.

| Metric | Result |
|---|---|
| Pixel Accuracy | 85.08% |
| Mean IoU | 53.59% |
| Mean Precision | 71.46% |
| Mean Recall | 65.25% |
| Mean F1 | 65.52% |
| Test Loss | 0.6832 |

Model selection was based on validation mIoU (best: 55.08%).

## Visual Results

### Qualitative Segmentation

![Qualitative Segmentation](results/qualitative_segmentation_results.png)

### Normalized Confusion Matrix

![Confusion Matrix](results/confusion_matrix_normalized.png)

### Per-Class Performance

![Per-Class Performance](results/per_class_performance.png)

### Metric Correlation

![Metric Correlation](results/metric_correlation_heatmap.png)

## Implementation

The complete experiment notebook is available at:

[DeepLabV3+ Xception Notebook](notebooks/DeepLabV3Plus_Xception_LIACi.ipynb)

The implementation follows the DeepLabV3+ architecture and uses the VainF implementation as an architectural reference:

https://github.com/VainF/DeepLabV3Plus-Pytorch

## Limitations

The model performs better on large structural regions than on difficult surface-condition classes. Defect and corrosion segmentation remain challenging.

## Future Work

- Class-imbalance-aware loss functions
- Improved augmentation strategies
- Further experiments to improve minority-class IoU
- Comparison with alternative segmentation architectures

## Notes

The dataset and trained checkpoint are not included in this repository.
    """

(repo / "README.md").write_text(readme, encoding="utf-8")

print("✓ README.md created")
print("Location:", repo / "README.md")